In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.listdir('/content/drive/MyDrive/Colab Notebooks/')

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks")
RAW_PQ = DATA_DIR
PROC = DATA_DIR

TRAIN_OUT = "train_fe.parquet"
VAL_OUT = "val_fe.parquet"
TEST_OUT = "test_fe.parquet"

RNG_SEED = 42
VAL_FRAC = 0.15

In [ ]:
# Get the 24 competitor feature column names
COMP_RATE = [f"comp{i}_rate" for i in range(1, 9)]
COMP_INV = [f"comp{i}_inv" for i in range(1, 9)]
COMP_DIFF = [f"comp{i}_rate_percent_diff" for i in range(1, 9)]

# Constants
EPS = 1e-4
BOOKING_WINDOW_MULT = 1000
LOC2_N_BUCKETS = 20


def add_datetime_features(df):
    # add temporal features
    dt = df["date_time"]
    df["month"] = dt.dt.month.astype("uint8")
    df["day_of_week"] = dt.dt.dayofweek.astype("uint8")
    df["hour"] = dt.dt.hour.astype("uint8")
    df["is_weekend"] = (dt.dt.dayofweek >= 5).astype("uint8")
    # absolute time in seconds since epoch (helps capture trends and seasonality,
    # and is easier for tree models to use than datetime)
    df["date_time_unix"] = (dt.astype("int64") // 10**9).astype("int64")


def add_price_features(df):
    # clip extreme prices (long right tail; some rows have $10k+) and take log to tame skew
    df["price_usd_clipped"] = df["price_usd"].clip(upper=5000.0).astype("float32")
    df["log_price"] = np.log1p(df["price_usd_clipped"]).astype("float32")

    # normalize price: some prices are total-stay, others per-night - dividing makes them comparable
    df["price_per_night"] = (
        df["price_usd_clipped"] / df["srch_length_of_stay"].clip(lower=1)
    ).astype("float32")

    # per-traveler cost
    pax = (
        df["srch_adults_count"].astype("float32") + df["srch_children_count"].astype("float32")
    ).clip(lower=1)
    df["price_per_person"] = (df["price_usd_clipped"] / pax).astype("float32")

    # listwise: where this hotel's price sits within the search results (1 = cheapest)
    grp = df.groupby("srch_id")["price_usd_clipped"]
    df["price_rank_in_query"] = grp.rank(method="average").astype("float32")

    # listwise: ratio to the search's mean price (above 1 = more expensive than average)
    query_mean = grp.transform("mean")
    df["price_rel_to_query_mean"] = (
        df["price_usd_clipped"] / query_mean.replace(0, np.nan)
    ).astype("float32")

    # is this listing cheaper than the hotel's typical price? negative = deal
    df["price_vs_prop_hist"] = (df["log_price"] - df["prop_log_historical_price"]).astype(
        "float32"
    )


def add_rank_features(df):
    grp = df.groupby("srch_id")
    # listwise rank within the search on each static field (1 = best in this search)
    df["star_rank_in_query"] = (
        grp["prop_starrating"].rank(method="average", ascending=False).astype("float32")
    )
    df["review_rank_in_query"] = (
        grp["prop_review_score"].rank(method="average", ascending=False).astype("float32")
    )
    df["loc1_rank_in_query"] = (
        grp["prop_location_score1"].rank(method="average", ascending=False).astype("float32")
    )
    df["loc2_rank_in_query"] = (
        grp["prop_location_score2"].rank(method="average", ascending=False).astype("float32")
    )


def add_competitor_features(df):
    # Aggregate 24 competitor features into a few dense features to reduce sparsity and noise
    rate = df[COMP_RATE].to_numpy(dtype="float32")
    inv = df[COMP_INV].to_numpy(dtype="float32")
    diff = df[COMP_DIFF].to_numpy(dtype="float32")

    # how many of the 8 competitors had any rate data (most cols are 90%+ NaN)
    df["comp_rate_n"] = (~np.isnan(rate)).sum(axis=1).astype("uint8")

    # sum across competitors: +1 = Expedia cheaper, 0 = tied, -1 = Expedia pricier
    df["comp_rate_sum"] = np.nansum(rate, axis=1).astype("float32")

    # sum of availability flags across competitors
    df["comp_inv_sum"] = np.nansum(inv, axis=1).astype("float32")

    # mean absolute % price difference vs competitors
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        diff_mean = np.nanmean(np.abs(diff), axis=1)
    df["comp_diff_mean"] = np.nan_to_num(diff_mean, nan=0.0).astype("float32")


def add_visitor_history_features(df):
    # 0/1 flag: does the visitor have any prior purchase history? (~95% don't)
    df["visitor_has_history"] = (~df["visitor_hist_starrating"].isna()).astype("uint8")

    # does this hotel match the visitor's usual bookings?
    df["star_diff_vs_visitor_hist"] = (
        df["prop_starrating"] - df["visitor_hist_starrating"]
    ).astype("float32")
    df["price_diff_vs_visitor_hist"] = (
        df["price_usd_clipped"] - df["visitor_hist_adr_usd"]
    ).astype("float32")

    # for users without history, fill diffs with 0 (effectively "no comparison")
    for c in [
        "visitor_hist_starrating",
        "visitor_hist_adr_usd",
        "star_diff_vs_visitor_hist",
        "price_diff_vs_visitor_hist",
    ]:
        df[c] = df[c].fillna(0).astype("float32")


def add_paper_composite_features(df):
    # Composite features from Liu et al. 2013 (https://arxiv.org/pdf/1311.7679)
    price = df["price_usd_clipped"]
    rooms = df["srch_room_count"].astype("float32")
    pax = (
        df["srch_adults_count"].astype("float32") + df["srch_children_count"].astype("float32")
    ).clip(lower=1)

    # dollar-space deal signal: how much cheaper than historical price
    df["ump"] = (np.exp(df["prop_log_historical_price"]) - price).astype("float32")

    # true transaction value (price * number of rooms booked)
    df["total_fee"] = (price * rooms).astype("float32")

    # true cost per traveler
    df["per_fee"] = (df["total_fee"] / pax).astype("float32")

    # location-quality x query-demand interaction
    df["score2ma"] = (df["prop_location_score2"] * df["srch_query_affinity_score"]).astype(
        "float32"
    )

    # ratio of the two location scores (epsilon avoids divide-by-zero)
    df["score1d2"] = (
        (df["prop_location_score2"] + EPS) / (df["prop_location_score1"] + EPS)
    ).astype("float32")

    # composite ID encoding (rooms, booking_window) jointly
    df["count_window"] = (
        rooms.astype("int32") * BOOKING_WINDOW_MULT + df["srch_booking_window"].astype("int32")
    ).astype("int32")


def add_count_features(df, prop_counts, dest_counts, loc2_bin_edges, loc2_bin_counts):
    # One-way count features from Liu et al. 2013 (https://arxiv.org/pdf/1311.7679)
    # how many times this hotel appears across train+val+test (popularity / long-tail signal)
    df["prop_id_1cnt"] = df["prop_id"].map(prop_counts).fillna(0).astype("uint32")

    # how many times this destination appears
    df["srch_destination_id_1cnt"] = (
        df["srch_destination_id"].map(dest_counts).fillna(0).astype("uint32")
    )

    # how common is this bucket of prop_location_score2 (20 quantile buckets)
    loc2_bin = np.digitize(df["prop_location_score2"].to_numpy(), loc2_bin_edges, right=False)
    df["prop_location_score2_1cnt"] = (
        pd.Series(loc2_bin).map(loc2_bin_counts).fillna(0).astype("uint32").to_numpy()
    )

In [ ]:
raw = pd.read_parquet(RAW_PQ / "train.parquet")

rng = np.random.default_rng(RNG_SEED)
all_srch = raw["srch_id"].unique()
rng.shuffle(all_srch)
n_val = int(len(all_srch) * VAL_FRAC)
val_ids = pd.Index(all_srch[:n_val])
is_val = raw["srch_id"].isin(val_ids)

val_raw = raw.loc[is_val].reset_index(drop=True)
train_raw = raw.loc[~is_val].reset_index(drop=True)
print(f"train_raw: {train_raw.shape[0]:,} rows, {train_raw['srch_id'].nunique():,} searches")
print(f"val_raw:   {val_raw.shape[0]:,} rows, {val_raw['srch_id'].nunique():,} searches")

In [ ]:
# Imputation values for the 4 high-missingness columns.
# Median is robust to skew (these columns all have long tails).
# Computed from train_raw only (fitting on train, applying to val and test).
median_fills = {
    "prop_review_score": float(train_raw["prop_review_score"].median()),
    "prop_location_score2": float(train_raw["prop_location_score2"].median()),
    "srch_query_affinity_score": float(train_raw["srch_query_affinity_score"].median()),
    "orig_destination_distance": float(train_raw["orig_destination_distance"].median()),
}

print("median fills:", median_fills)

# Load test now so we can build count lookups over train+val+test combined.
# Count features use no labels - just feature frequencies - so this is not leakage.
test_raw = pd.read_parquet(RAW_PQ / "test.parquet")
print(f"test_raw:  {test_raw.shape[0]:,} rows, {test_raw['srch_id'].nunique():,} searches")

# Count lookups: how often each prop_id / destination_id appears across all splits.
# transform() will use .map() to stamp each row with its global frequency.
prop_counts = pd.concat([
    train_raw["prop_id"], val_raw["prop_id"], test_raw["prop_id"]
]).value_counts()
dest_counts = pd.concat([
    train_raw["srch_destination_id"], val_raw["srch_destination_id"], test_raw["srch_destination_id"]
]).value_counts()

# Bucketed count for prop_location_score2 (a continuous float).
# 1. Concatenate all splits, fill NaN with the train median.
# 2. Compute 20 equal-frequency quantile cut points (drops the 0% and 100% endpoints).
# 3. np.digitize assigns each row to a bucket id 0..20.
# 4. Count rows per bucket - by construction roughly uniform across buckets.
loc2_all = pd.concat([
    train_raw["prop_location_score2"], val_raw["prop_location_score2"], test_raw["prop_location_score2"]
]).fillna(median_fills["prop_location_score2"]).to_numpy()
loc2_bin_edges = np.quantile(loc2_all, np.linspace(0, 1, LOC2_N_BUCKETS + 1))[1:-1]
loc2_all_bin = np.digitize(loc2_all, loc2_bin_edges, right=False)
loc2_bin_counts = pd.Series(loc2_all_bin).value_counts()

print(f"prop_id_1cnt:     {len(prop_counts):,} unique props,  median count = {prop_counts.median():.0f}")
print(f"dest_id_1cnt:     {len(dest_counts):,} unique dests,  median count = {dest_counts.median():.0f}")
print(f"loc2_bin_counts:  {len(loc2_bin_counts):,} buckets,    range = [{loc2_bin_counts.min()}, {loc2_bin_counts.max()}]")

In [ ]:
def transform(df, *, is_train):
    for col, val in median_fills.items():
        df[col] = df[col].fillna(val).astype("float32")

    add_datetime_features(df)
    add_price_features(df)
    add_rank_features(df)
    add_competitor_features(df)
    add_visitor_history_features(df)
    add_paper_composite_features(df)
    add_count_features(df, prop_counts, dest_counts, loc2_bin_edges, loc2_bin_counts)

    if is_train:
        df["relevance"] = np.where(
            df["booking_bool"] == 1,
            5,
            np.where(df["click_bool"] == 1, 1, 0),
        ).astype("int8")

    df = df.drop(columns=COMP_RATE + COMP_INV + COMP_DIFF)
    return df

In [ ]:
train_fe = transform(train_raw, is_train=True)
print(f"train_fe: {train_fe.shape}")
print("NaNs remaining:", int(train_fe.isna().sum().sum()))
train_fe.to_parquet(PROC / TRAIN_OUT, compression="snappy", index=False)
print(f"  -> wrote {(PROC / TRAIN_OUT).stat().st_size / (1 << 20):.1f} MB")

In [ ]:
val_fe = transform(val_raw, is_train=True)
print(f"val_fe: {val_fe.shape}")
val_fe.to_parquet(PROC / VAL_OUT, compression="snappy", index=False)
print(f"  -> wrote {(PROC / VAL_OUT).stat().st_size / (1 << 20):.1f} MB")

In [ ]:
test_fe = transform(test_raw, is_train=False)
print(f"test_fe: {test_fe.shape}")
test_fe.to_parquet(PROC / TEST_OUT, compression="snappy", index=False)
print(f"  -> wrote {(PROC / TEST_OUT).stat().st_size / (1 << 20):.1f} MB")

In [ ]:
for name, fname in [("train", TRAIN_OUT), ("val", VAL_OUT), ("test", TEST_OUT)]:
    p = PROC / fname
    if p.exists():
        df = pd.read_parquet(p, columns=["srch_id"])
        print(f"{name}: {p.stat().st_size / (1 << 20):>6.1f} MB  |  {len(df):>9,} rows  |  {df['srch_id'].nunique():>7,} searches")